# El arte del lúpulo: datos, estilos y valoraciones de las cervezas.
### Análisis exploratorio de datos (EDA).

Los datos proceden del conjunto de datos [Beer Reviews de Kaggle](https://www.kaggle.com/datasets/rdoume/beerreviews), que contiene aproximadamente 1,59 millones de reseñas de cervezas. 
Cada fila representa una reseña individual, por lo que una misma cerveza puede aparecer en varias filas.

El objetivo del proyecto es analizar qué características se relacionan con las valoraciones de las cervezas, identificar los estilos mejor valorados y estudiar si existe una relación entre la graduación alcohólica y la puntuación general. 

Los resultados servirán como base para desarrollar una aplicación que permita explorar y recomendar cervezas según las preferencias del usuario.

## 0. Comprobación

In [ ]:
import pandas as pd
import matplotlib

print("Todo listo")

## 1. ¿Qué tengo?

Empieza por ver qué hay en la tabla.

**1.1** Carga tu CSV de la carpeta datos en una tabla con un nombre que diga qué es (como pisos o titanic) y enseña las primeras filas.

In [ ]:
cervezas = pd.read_csv(
    "datos/beer_reviews.csv",
    usecols=["review_overall", "beer_style", "beer_name", "beer_abv", "brewery_name"]
)

# Dejamos las columnas en el orden que vamos a utilizar.
cervezas = cervezas[["review_overall", "beer_style", "beer_name", "beer_abv", "brewery_name"]]

cervezas.head()


**1.2** ¿Cuántas filas y columnas tiene?

In [ ]:
cervezas.shape

**1.3** Mira el info(). ¿De qué tipo es cada columna? ¿Cuáles tienen huecos?

In [ ]:
cervezas.info()

# Cargamos únicamente cinco columnas de las trece originales.
# Los valores ausentes de graduación se tratarán en el apartado 2.


**1.4** Quédate con las columnas que te sirven para tu pregunta, o quita las que sobran. En este caso, seleccionamos cinco columnas al cargar el archivo.

In [ ]:
# Ya hemos seleccionado las cinco columnas necesarias al cargar el archivo.
cervezas.columns


**1.5** Si los nombres de las columnas están en inglés o son raros, ponles nombres en español.

In [ ]:
cervezas = cervezas.rename(columns={
    "review_overall": "rating",
    "beer_style": "estilo_cerveza",
    "beer_name": "cerveza",
    "beer_abv": "graduacion",
    "brewery_name": "cerveceria"
})

cervezas.columns


Lo que he visto y lo que he decidido:

El conjunto de datos contiene reseñas de cervezas. Cada fila representa una reseña individual, por lo que una misma cerveza puede aparecer varias veces.

He seleccionado desde la carga cinco columnas útiles para el proyecto: valoración general, estilo, nombre de la cerveza, graduación alcohólica y cervecería. He cambiado sus nombres al español para trabajar con ellas de forma más clara.

## 2. ¿Qué me falta?

Cuenta lo que falta y decide qué haces con cada columna.

**2.1** Cuenta los huecos de cada columna.

In [ ]:
cervezas.isna().sum()

# .isna() comprueba los valores ausentes y devuelve True si falta dato.
# .sum() cuenta los valores ausentes de cada columna.


**2.2** Calcula el porcentaje de huecos de cada columna.

In [ ]:
porcentaje = cervezas.isna().sum() / len(cervezas) * 100
porcentaje.round(2)


**2.3** Decide qué haces con cada columna que tenga huecos: quitar la columna, quitar esas filas o rellenarlos. Hazlo.

In [ ]:
cervezas = cervezas.dropna(subset=["graduacion"])
cervezas.shape

# Eliminamos las filas que no tienen graduación alcohólica,
# porque necesitamos ese dato para estudiar su relación con la valoración.


**2.4** Comprueba que ya no quedan huecos, o que los que quedan los dejas a propósito.

In [ ]:
cervezas.isna().sum()

**2.5** Mira con value_counts las columnas de texto. ¿Hay algún hueco disfrazado, como "Unknown", "-" o "N/A"?

In [ ]:
# value_counts() cuenta cuántas veces aparece cada valor.

print(cervezas["cerveceria"].value_counts())
print(cervezas["estilo_cerveza"].value_counts())
print(cervezas["cerveza"].value_counts())


Lo que he visto y lo que he decidido:

He cargado únicamente las cinco columnas necesarias para el análisis y he cambiado sus nombres al español. También he eliminado las filas que no tenían datos de graduación alcohólica, porque necesito esa información para comparar la valoración según el contenido de alcohol.

Quedan algunos valores ausentes en el nombre de la cervecería. Los mantengo porque no afectan a los análisis principales.

## 3. ¿Hay filas repetidas?

Comprueba si hay algo contado dos veces.

**3.1** ¿Cuántas filas repetidas hay?

In [ ]:
print("Filas repetidas:", cervezas.duplicated().sum())

**3.2** Si tienes una columna que identifica cada fila (un id, un nombre), ¿hay algún valor repetido?

In [ ]:
# Comprobamos cuántos nombres de cerveza aparecen más de una vez.
print("Nombres de cervezas repetidos:", cervezas["cerveza"].duplicated().sum())


**3.3** Decide si las quitas, y hazlo. Si no son la misma fila, no se quitan.

In [ ]:
# No eliminamos automáticamente las filas repetidas.
# Como solo conservamos cinco columnas, dos reseñas distintas pueden coincidir
# en todos estos datos, aunque originalmente fueran reseñas diferentes.
print("Filas repetidas en las cinco columnas:", cervezas.duplicated().sum())


In [ ]:
# Conservamos las filas repetidas porque cada fila representa una reseña.
# Con las cinco columnas elegidas no podemos confirmar si dos filas iguales
# proceden de la misma reseña o de reseñas diferentes.


Lo que he visto y lo que he decidido:

Hay nombres de cervezas repetidos, algo normal porque una cerveza puede recibir varias reseñas.

Al trabajar solo con cinco columnas, algunas filas pueden coincidir en todos sus valores aunque procedan de reseñas distintas. Por eso no elimino automáticamente las filas repetidas: no tengo información suficiente para asegurar que sean duplicados reales.

## 4. ¿Cómo son los números?

Mira los números con el describe, la media, la mediana y un histograma.

**4.1** Haz el describe. ¿Qué te llama la atención?

In [ ]:
# Resumen estadístico de las dos columnas numéricas del análisis.
cervezas[["rating", "graduacion"]].describe().round(2)

# La valoración general tiene una media de 3,82 sobre 5.
# La graduación media es del 7,04 %, con un máximo de 57,7 %.


**4.2** Saca la media y la mediana de la columna de números más importante para tu pregunta. ¿Se parecen?

In [ ]:
print("Valoración media:", round(cervezas["rating"].mean(), 2))
print("Valoración mediana:", round(cervezas["rating"].median(), 2))

# La media y la mediana son parecidas y apuntan a valoraciones generalmente altas.


**4.3** Dibuja su histograma.

In [ ]:
cervezas["rating"].plot(kind="hist", bins=50, title="Distribución de las valoraciones de las cervezas")

# .plot(kind="hist"): dibujamos un histograma.
# bins=50: dividimos los valores en 50 intervalos.

**4.4** Haz lo mismo con otra columna de números.

In [ ]:
print("Graduación media:", round(cervezas["graduacion"].mean(), 2))
print("Graduación mediana:", round(cervezas["graduacion"].median(), 2))

# La media (7,04 %) es superior a la mediana (6,5 %).


In [ ]:
cervezas["graduacion"].plot(kind="hist", bins=20, title="Distribución de la graduación de las cervezas")

# El histograma apoya esta teoría: la mayoría de los registros se concentra en graduaciones relativamente bajas, mientras que hay unos pocos valores mucho más altos.

Lo que he visto y lo que he decidido:

El análisis estadístico muestra que las valoraciones son generalmente altas, con una puntuación media de 3,82 sobre 5. La graduación presenta una media del 7,04 % y una mediana del 6,5 %, lo que sugiere que algunos valores elevados aumentan la media.

El histograma permite observar cómo se distribuyen las valoraciones y las graduaciones antes de estudiar su relación.

## 5. ¿Hay algo raro?

Busca los valores extremos y decide si son errores o casos reales.

**5.1** Enseña las 5 filas con el valor más alto de tu columna principal.

In [ ]:
cervezas.sort_values("rating", ascending=False).head()

# sort_values("rating") ordena las reseñas por valoración.
# ascending=False las ordena de mayor a menor.


**5.2** Y las 5 con el más bajo. ¿Hay ceros o valores imposibles?

In [ ]:
cervezas.sort_values("rating").head()

# Revisamos las reseñas con las valoraciones más bajas.


**5.3** Haz el diagrama de caja de esa columna.

In [ ]:
# El diagrama de caja ayuda a observar la distribución y los valores extremos.
cervezas["rating"].plot(kind="box", title="Distribución de las valoraciones de las cervezas")


**5.4** Decide qué haces con cada valor raro: si es un error, corrígelo o quítalo; si es real, se queda.

In [ ]:
# Las valoraciones de 0 están dentro de la escala observada.
# Las conservamos porque no tenemos pruebas suficientes para considerarlas errores.


Lo que he visto y lo que he decidido:

Las valoraciones generales oscilan entre 0 y 5, con algunos valores bajos que aparecen como posibles valores atípicos en el diagrama de caja.

He decidido conservar las puntuaciones de 0 y las valoraciones bajas, ya que forman parte de la escala observada y no tengo pruebas suficientes para afirmar que sean errores.

## 6. ¿Qué hay más?

Cuántas filas hay de cada tipo.

**6.1** Haz el value_counts de la columna de texto más importante, en número y en porcentaje.

In [ ]:
cervezas["estilo_cerveza"].value_counts() # para el recuento absoluto

In [ ]:
(cervezas["estilo_cerveza"].value_counts(normalize=True) * 100).round(2)    # para el recuento en porcentaje

# value_counts(normalize=True): calcula la proporción de cada estilo.


**6.2** Dibújalo en barras.

In [ ]:
# Vamos a representar los 10 estilos de cerveza con más reseñas:
cervezas["estilo_cerveza"].value_counts().head(10).plot(kind="bar", title="Los 10 estilos de cerveza con más reseñas")

Lo que he visto y lo que he decidido:

Los estilos de cerveza no están representados por igual: American IPA concentra el mayor porcentaje de reseñas (7,45 %), seguido de American Double / Imperial IPA (5,60 %) y American Pale Ale (APA) (3,82 %). 
En total aparecen 104 estilos, con una presencia muy desigual entre ellos.
He decidido analizar los diez estilos más frecuentes en el gráfico para facilitar la comparación y conservar todos los estilos en el conjunto de datos, ya que los menos frecuentes también pueden ser relevantes para las recomendaciones de la aplicación.

## 7. ¿Cambia algo de un grupo a otro?

Aquí es donde suelen salir los hallazgos.

**7.1** Con groupby, calcula la media o la mediana de tu columna principal en cada grupo.

In [ ]:
# Queremos investigar si la valoración general (rating) cambia según el estilo de cerveza (estilo_cerveza). 
# Empezamos identificando qué estilos reciben mejores puntuaciones, de mayor a menor

cervezas.groupby("estilo_cerveza")["rating"].mean().round(2).sort_values(ascending=False)

**7.2** Haz otra comparación con otra columna de grupos.

In [ ]:
# Investigamos si la valoración general (rating) cambia según la graduación alcohólica.
# Agrupamos las cervezas por intervalos de graduación para comparar sus valoraciones medias.

cervezas.groupby(pd.cut(cervezas["graduacion"], bins=[0, 5, 7, 9, 60], right=True))["rating"].mean().round(2)

**7.3** Antes de sacar conclusiones, mira cuántas filas tiene cada grupo. ¿Hay alguno con muy pocas?

In [ ]:
# Contamos cuántas reseñas hay en cada intervalo de graduación.

print("Graduación hasta el 5 %:", len(cervezas[cervezas["graduacion"] <= 5]))
print("Graduación entre más del 5 % y el 7 %:", len(cervezas[(cervezas["graduacion"] > 5) & (cervezas["graduacion"] <= 7)]))
print("Graduación entre más del 7 % y el 9 %:", len(cervezas[(cervezas["graduacion"] > 7) & (cervezas["graduacion"] <= 9)]))
print("Graduación superior al 9 %:", len(cervezas[cervezas["graduacion"] > 9)])


Lo que he visto y lo que he decidido:

Las valoraciones medias varían según el estilo de cerveza. Entre los estilos con mejores puntuaciones se encuentran American Wild Ale (4,10), Gueuze (4,09) y Quadrupel (4,07), mientras que Low Alcohol Beer obtiene una media de 2,55.

También se observa una diferencia según la graduación: las cervezas de hasta un 5 % obtienen una media de 3,58, frente a las de más de un 9 %, cuya media es de 3,94. Todos los grupos tienen un número elevado de reseñas.

Estos resultados sugieren que el estilo y la graduación pueden estar relacionados con la valoración, pero una graduación mayor no implica necesariamente una mejor puntuación.

## 8. ¿Van juntas dos columnas?

Comprueba si dos columnas de números tienen relación.

**8.1** Calcula la correlación entre dos columnas de números.

In [ ]:
cervezas["graduacion"].corr(cervezas["rating"])

**8.2** Haz el gráfico de puntos de esas dos columnas.

In [ ]:
cervezas.plot(kind="scatter", x="graduacion", y="rating", title="Relación entre graduación y valoración")

Lo que he visto y lo que he decidido:

La correlación es de aproximadamente 0,14, lo que indica una relación positiva débil. 

Aunque las cervezas con mayor graduación tienden a recibir valoraciones algo superiores, el gráfico muestra que hay puntuaciones muy diferentes para graduaciones similares, por lo tanto, la graduación alcohólica por sí sola no explica la valoración de una cerveza.

## 9. ¿Qué he aprendido?

Las tres cosas que he podido descubrir tras realizar el EDA han sido:

1. Distribución de las valoraciones: la valoración general presenta una media de 3,82 y una mediana de 4 sobre 5.

2. Relación entre graduación y valoración: la correlación de Pearson es aproximadamente 0,14, lo que indica una asociación lineal positiva débil. La graduación por sí sola no explica las diferencias entre puntuaciones.

3. Diferencias entre grupos: la valoración media varía según el estilo, desde 2,55 en Low Alcohol Beer hasta 4,10 en American Wild Ale. También aumenta de 3,58 en las cervezas de hasta un 5 % de alcohol a 3,94 en las de más de un 9 %.

Para preparar los datos he cargado solo cinco de las trece columnas originales, he cambiado sus nombres al español y he eliminado las filas que no tenían información sobre la graduación alcohólica.

El conjunto original contiene 1.586.614 reseñas. Tras eliminar las filas sin graduación, quedan 1.518.829 reseñas.

## 10. Guardar los datos limpios

**10.1** Guarda en `datos/datos_limpios.csv` las columnas que necesita la aplicación Streamlit, usando `to_csv` e `index=False`.

El archivo exportado contiene el nombre de la cerveza, su estilo, la graduación alcohólica, la valoración general y la cervecería. `index=False` evita guardar el índice de Pandas como una columna adicional.

In [ ]:
cervezas[["cerveza", "estilo_cerveza", "graduacion", "rating", "cerveceria"]].to_csv(
    "datos/datos_limpios.csv",
    index=False
)
